# Ensemble Methods & Hyperparameter Tuning
## Credit Risk Dataset — Practice Notebook

**Dataset**: UCI *Default of Credit Card Clients* — Yeh & Lien (2009)  
**Target**: `DEFAULT_NEXT_MONTH` — binary: 1 = default, 0 = no default  
**Class balance**: ~22% default (minority class) — all models must handle this explicitly

---

**Covers**: Bagging · Boosting · Stacking · Voting · Log Loss · ROC-AUC · Random Search · Genetic Algorithm · Bayesian Optimisation

---
## 1. Imports

**What and why:**

- `pathlib.Path` — cross-platform file path handling; avoids hardcoded OS-specific separators.
- `scipy.stats.randint` — generates integer ranges for `RandomizedSearchCV` parameter distributions.
- `skopt` (scikit-optimize) — provides `BayesSearchCV` and `gp_minimize` for Bayesian hyperparameter optimisation using Gaussian Processes.
- `random` — used by the custom Genetic Algorithm to create and mutate individuals in the parameter population.
- All sklearn ensemble imports are collected here so the notebook is self-contained and dependencies are visible at a glance.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import random
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, BaggingClassifier,
    GradientBoostingClassifier, StackingClassifier, VotingClassifier
)
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    roc_curve, log_loss, classification_report,
    precision_score, recall_score
)
from scipy.stats import randint
import matplotlib.pyplot as plt
import sys
print(sys.executable)

from skopt import gp_minimize
from skopt.space import Integer
from skopt.utils import use_named_args
# Bayesian optimisation — install with:
# c:\Users\Kasia\AppData\Local\Programs\Python\Python311\python.exe -m pip install hyperopt
#from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval

pd.set_option("display.max_columns", None)

---
## 2. Load & Prepare Data

**What and why:**

The UCI Credit Card Default dataset is stored as an Excel file with a description row at index 0 and the actual header at index 1 — hence `header=1`. The `xlrd` engine is required for `.xls` format (the newer `openpyxl` engine only reads `.xlsx`).

**Category fixes applied before encoding:**
- `EDUCATION`: values 0, 5, 6 are not documented in the original paper — collapsed to 4 (Other).
- `MARRIAGE`: value 0 is undocumented — collapsed to 3 (Other).

**Why stratified split matters:**  
With a 22%/78% default/non-default ratio, an unstratified split can produce train and test sets with meaningfully different default rates, making evaluation unstable. `stratify=y` guarantees both sets have the same proportion.

**Why two versions of X (scaled vs unscaled):**  
- `X_train_sc` / `X_test_sc` — for distance-based and linear models (KNN, SVM, Logistic Regression) that are sensitive to feature magnitude.
- `X_train` / `X_test` — for tree-based models (Random Forest, XGBoost, Gradient Boosting) that are scale-invariant and use raw split values.

In [ ]:
# ── paths ──────────────────────────────────────────────────────────────────
REPO_ROOT  = Path(r"F:\Apps\Credit-Risk-Score-ML")
DATA_PATH  = REPO_ROOT / "data" / "raw" / "default of credit card clients.xls"
RANDOM_STATE = 42

print("Data path:", DATA_PATH)
print("Exists:   ", DATA_PATH.exists())

In [ ]:
df_raw = pd.read_excel(DATA_PATH, sheet_name=0, engine="xlrd", header=1)
df = df_raw.rename(columns={"default payment next month": "DEFAULT_NEXT_MONTH"})

# fix undocumented category codes
df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"]  = df["MARRIAGE"].replace({0: 3})

print("Shape:", df.shape)
print("\nTarget distribution (%):")
print((df["DEFAULT_NEXT_MONTH"].value_counts(normalize=True) * 100).round(2))

In [ ]:
# ── features & target ───────────────────────────────────────────────────────
TARGET = "DEFAULT_NEXT_MONTH"

# Drop ID — not a predictive feature
X = df.drop(columns=["ID", TARGET], errors="ignore")
y = df[TARGET].astype(int)

# One-hot encode the three categorical columns
X = pd.get_dummies(X, columns=["SEX", "EDUCATION", "MARRIAGE"], drop_first=True)

# Train / test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale — important for KNN, SVM, Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print("Train:", X_train.shape, "  Test:", X_test.shape)
print("Default rate — train:", y_train.mean().round(4),
      "  test:", y_test.mean().round(4))

---
## 3. Evaluation Helper

**What and why:**

The dataset is imbalanced (~22% default), so **ROC-AUC and Macro F1** are better measures than accuracy alone. A naive classifier that always predicts "no default" achieves 78% accuracy without learning anything useful — accuracy alone would make such a model look good.

- **Accuracy** — included for reference but never used as the primary selection criterion.
- **Macro F1** — harmonic mean of precision and recall, computed equally across both classes. Penalises models that perform well on the majority class but poorly on the minority (default) class.
- **ROC-AUC** — measures the model's ability to rank defaulters above non-defaulters across all thresholds. Standard metric in credit risk model validation (Basel II, EBA RTS).

**Credit risk benefit:** standardised output across all models allows direct comparison in the summary table in Section 11, without re-writing evaluation logic per model.

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name, scaled=False):
    
    model.fit(X_tr, y_tr)
    preds  = model.predict(X_te)
    probs  = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else None

    acc  = accuracy_score(y_te, preds)
    f1   = f1_score(y_te, preds, average="macro")
    auc  = roc_auc_score(y_te, probs) if probs is not None else None

    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Macro-F1  : {f1:.4f}")
    if auc:
        print(f"  ROC-AUC   : {auc:.4f}")
    print(classification_report(y_te, preds, target_names=["No Default", "Default"]))
    return auc or acc

---
## 4. Bagging

### What is Bagging?

**Bagging** (Bootstrap Aggregating) trains multiple copies of the same base model on different random subsets of the training data, sampled *with replacement* (bootstrapping). Predictions are combined by majority vote (classification) or averaging (regression).

### Why use it in ML?

- Reduces **variance** — individual high-variance models (KNN, deep trees) overfit their training subset; averaging across 30 bootstrapped models smooths this out.
- Each bootstrap sample covers ~63% of the data; the remaining ~37% (out-of-bag samples) can be used for internal validation at no extra cost.
- Parallelisable — each base model trains independently.

### How does it benefit credit risk prediction?

- KNN classifies a borrower based on the k most similar borrowers in the training set. Credit data is noisy (BILL_AMT outliers, irregular PAY_X sequences) — single KNN is unstable. Bagging over 30 bootstraps stabilises it significantly.
- Bagging with Logistic Regression introduces diversity through data resampling even though LR itself is a low-variance model — useful as a stable, interpretable baseline ensemble.
- `class_weight="balanced"` in the LR base estimator ensures each bootstrap sample corrects for the 22%/78% imbalance, preventing the ensemble from being dominated by the majority class.

### Requirements

- Base estimator must implement `.fit()` and `.predict_proba()`.
- `n_estimators`: 20–100 typical. Returns stabilise around 50.
- **Scaling required** for distance-based and linear base models — pass `X_train_sc` / `X_test_sc`.
- `class_weight` is not a parameter of `BaggingClassifier` itself — set it on the base estimator.

In [ ]:
base_knn = KNeighborsClassifier(n_neighbors=5)

bagging_knn = BaggingClassifier(
    estimator=base_knn,
    n_estimators=30,
    random_state=RANDOM_STATE
)

# KNN needs scaled data
evaluate_model(bagging_knn, X_train_sc, y_train, X_test_sc, y_test, "Bagging — KNN (scaled)")

In [ ]:
# Try bagging on Logistic Regression — handles imbalance with class_weight
base_lr = LogisticRegression(max_iter=1000, class_weight="balanced")

bagging_lr = BaggingClassifier(
    estimator=base_lr,
    n_estimators=30,
    random_state=RANDOM_STATE
)

evaluate_model(bagging_lr, X_train_sc, y_train, X_test_sc, y_test, "Bagging — Logistic Regression (scaled)")

---
## 5. Boosting

### What is Boosting?

**Boosting** trains models **sequentially**. Each new model focuses on the errors of the previous one — misclassified samples receive higher weights so the next learner pays more attention to them. Final predictions combine all models weighted by their individual accuracy.

Key difference from Bagging: Bagging trains models **in parallel** on random subsets; Boosting trains models **in sequence** to correct accumulated errors.

---
### 5a. XGBoost

### What is XGBoost?

**XGBoost** (Extreme Gradient Boosting) is a regularised gradient boosting implementation that adds L1/L2 penalties to the tree-building objective, column subsampling per tree, and hardware-optimised parallelism. It consistently outperforms standard sklearn Gradient Boosting on tabular data.

### Why use it in ML?

- Built-in `scale_pos_weight` parameter directly addresses class imbalance without oversampling.
- Column subsampling adds diversity, reducing overfitting on high-cardinality feature sets.
- Native SHAP support — `shap.TreeExplainer` can decompose any XGBoost prediction into per-feature contributions in O(TLD) time.
- `eval_metric="logloss"` aligns training objective with the probability calibration goal.

### How does it benefit credit risk prediction?

- `scale_pos_weight = neg/pos ≈ 3.5` tells XGBoost to weight default cases 3.5× more heavily during training — directly compensating for the 22%/78% split without altering the dataset.
- Captures non-linear interactions between PAY_0 (most recent repayment delay) and LIMIT_BAL (credit limit) that logistic regression misses.
- In Lessmann et al. (2015) benchmark, gradient boosting variants consistently ranked among the top performers on credit scoring tasks.
- SHAP output satisfies EU AI Act Article 13 transparency requirements: each loan decision can be explained with a waterfall chart.

### Requirements

- `scale_pos_weight`: compute as `(y_train == 0).sum() / (y_train == 1).sum()` ≈ 3.5 for this dataset.
- Feature scaling **not required** — tree splits use rank-order information only.
- Pass unscaled `X_train` / `X_test`.

In [ ]:
# scale_pos_weight handles class imbalance in XGBoost
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight = {spw:.2f}  (neg/pos ratio)")

xgb_model = xgb.XGBClassifier(
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    scale_pos_weight=spw
)

evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "XGBoost")

### 5b. Gradient Boosting

### What is Gradient Boosting?

**Gradient Boosting** (sklearn implementation) builds an additive ensemble of shallow decision trees where each new tree fits the **pseudo-residuals** (negative gradient of the loss function) of the current ensemble. Unlike XGBoost it does not have built-in regularisation or column subsampling by default.

### Why use it in ML?

- Solid, well-understood baseline for tabular classification before tuning XGBoost or LightGBM.
- `learning_rate` controls how much each tree contributes — lower rate + more trees = better generalisation (but slower training).
- No external dependencies — part of sklearn core.

### How does it benefit credit risk prediction?

- Useful as a **comparison baseline** against XGBoost: if both achieve similar AUC, XGBoost is preferred in production due to SHAP support and better regulatory explainability tooling.
- `n_estimators=100` with `learning_rate=0.1` is a well-established starting configuration for credit risk datasets of this size.
- Note: sklearn GB has no `scale_pos_weight` — imbalance is handled via `sample_weight` in `.fit()` or by adjusting the classification threshold post-training.

### Requirements

- No `scale_pos_weight` — pass `sample_weight=compute_sample_weight("balanced", y_train)` to `.fit()` if needed.
- Feature scaling **not required**.
- Slower than XGBoost for large datasets — consider `HistGradientBoostingClassifier` for >50k rows.

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=RANDOM_STATE
)

evaluate_model(gb_model, X_train, y_train, X_test, y_test, "Gradient Boosting")

---
## 6. Stacking

### What is Stacking?

**Stacking** (Stacked Generalisation) trains multiple diverse **base models** (level-0 learners), then trains a **meta-model** (level-1 learner) on the base models' predictions. The meta-model learns the optimal way to combine base model outputs — unlike Voting, which uses a fixed rule.

Scikit-learn uses **cross-validation internally** (default cv=5) to generate the base model predictions used to train the meta-model, preventing data leakage from training to meta-training.

### Why use it in ML?

- Captures complementary strengths of different model families — Random Forest captures non-linear feature interactions, XGBoost captures sequential repayment patterns, and the meta-model LR combines both optimally.
- The meta-model can learn to down-weight a weaker base model and amplify a stronger one — automatically.
- Often achieves the highest performance of any single approach when base models are sufficiently diverse.

### How does it benefit credit risk prediction?

- Credit default is driven by diverse signal types: PAY_X repayment sequences (tree models), BILL_AMT balance trends (linear models), and LIMIT_BAL credit utilisation (captured differently by each family). Stacking integrates all of them in a principled way.
- Using Logistic Regression as the meta-model keeps the combination layer interpretable — regulators can inspect the coefficients to understand how RF and XGBoost scores are weighted.
- The stacking result vs the best individual model quantifies the **ensemble premium** — useful for justifying the additional complexity in the thesis.

### Requirements

- Base models must be diverse — combining two near-identical models adds little value.
- All base models must support `predict_proba` for probability-based stacking.
- Computationally expensive: cv=5 × number of base models × full training cost per fold.
- `class_weight="balanced"` must be set on each base model individually — `StackingClassifier` does not propagate it.

In [ ]:
base_models = [
    ("rf",  RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=RANDOM_STATE)),
    ("xgb", xgb.XGBClassifier(eval_metric="logloss", scale_pos_weight=spw, random_state=RANDOM_STATE)),
]
meta_model = LogisticRegression(max_iter=1000, class_weight="balanced")

stack_clf = StackingClassifier(estimators=base_models, final_estimator=meta_model)
evaluate_model(stack_clf, X_train, y_train, X_test, y_test, "Stacking — RF + XGB → LR")

---
## 7. Voting Ensemble

### What is a Voting Ensemble?

A **Voting Classifier** trains multiple models independently (no sequential dependency, no meta-model) and combines their predictions using a fixed rule:

- **Hard Voting** — each model casts a class vote; majority wins. Simple, no probabilities required.
- **Soft Voting** — each model outputs class probabilities; these are averaged across models. The class with the highest average probability wins. Generally more accurate than hard voting when models produce well-calibrated probabilities.

---
### 7a. Hard Voting

### Why use it in ML?

- Works even when base models do not support `predict_proba` (e.g. non-probability SVM).
- Simple to explain: the most popular answer among three models wins.
- Reduces variance by averaging out individual model errors.

### How does it benefit credit risk prediction?

- Combining LR (interpretable, linear), RF (non-linear, robust), and GaussianNB (fast probabilistic) gives a diverse committee that covers different assumptions about the data distribution.
- GaussianNB assumes feature independence but is very fast — useful as a lightweight third vote that introduces diversity without high computational cost.
- Hard voting can be presented to non-technical stakeholders as a simple democratic decision: three models vote, majority decides.

### Requirements

- Only `.predict()` is needed from each base model.
- Scaled input required for LR (scale-sensitive) — pass `X_train_sc` / `X_test_sc`.
- RF and GNB are scale-invariant but accept scaled input without issue.

In [ ]:
clf_lr  = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
clf_rf  = RandomForestClassifier(n_estimators=50, class_weight="balanced", random_state=RANDOM_STATE)
clf_gnb = GaussianNB()

hard_voting = VotingClassifier(
    estimators=[("lr", clf_lr), ("rf", clf_rf), ("gnb", clf_gnb)],
    voting="hard"
)

evaluate_model(hard_voting, X_train_sc, y_train, X_test_sc, y_test, "Hard Voting — LR + RF + GNB")

### 7b. Soft Voting

### Why use it in ML?

- Averages predicted **probabilities** instead of binary votes — confident predictions carry more weight than uncertain ones.
- Generally outperforms hard voting when all base models are well-calibrated.
- Produces a continuous probability output suitable for threshold tuning.

### How does it benefit credit risk prediction?

- The averaged probability score is a **credit risk score** — directly usable as a PD estimate fed into Expected Loss calculations: `EL = PD × LGD × EAD`.
- A lender can set a custom threshold (e.g. flag top 15% highest probability scores for manual underwriting review) rather than relying on the binary 0.5 cut-off.
- In production, soft voting output is more stable across slightly different data samples than hard voting — important for model monitoring and PSI (Population Stability Index) drift detection.

### Requirements

- All base models must support `.predict_proba()`. GaussianNB does natively; SVC requires `probability=True`.
- Scaled input required for LR — same `X_train_sc` / `X_test_sc` as hard voting above.

In [ ]:
soft_voting = VotingClassifier(
    estimators=[("lr", clf_lr), ("rf", clf_rf), ("gnb", clf_gnb)],
    voting="soft"
)

evaluate_model(soft_voting, X_train_sc, y_train, X_test_sc, y_test, "Soft Voting — LR + RF + GNB")

---
## 8. Log Loss

### What is Log Loss?

**Log Loss** (Binary Cross-Entropy) measures the quality of **predicted probabilities**, not just class labels. It penalises confident wrong predictions exponentially — a model that assigns 99% probability to no-default when the borrower actually defaults receives a very high (bad) log loss.

`L = -1/N × Σ [y·log(p) + (1-y)·log(1-p)]` — lower is better.

### Why use it in ML?

- Directly optimised by Logistic Regression and XGBoost (`eval_metric="logloss"`).
- Complements ROC-AUC: a model can rank defaulters well (high AUC) but produce unreliable probability estimates (high log loss). Both need to be checked.
- Essential when downstream decisions depend on the probability value, not just the binary prediction.

### How does it benefit credit risk prediction?

- Credit decisioning uses PD scores — not binary labels — to set interest rates, credit limits, and regulatory capital reserves (Basel II IRB Advanced approach: `EL = PD × LGD × EAD`).
- A model with low log loss produces **well-calibrated** PD scores: if the model says 30% chance of default, roughly 30% of borrowers at that score level should actually default. This calibration is required for IRB regulatory approval.
- Comparing log loss between LR and XGBoost here shows whether the more complex model also produces better-calibrated probabilities — not just better rankings.

In [ ]:
lr_for_loss = LogisticRegression(max_iter=1000, class_weight="balanced")
lr_for_loss.fit(X_train_sc, y_train)
y_prob_lr = lr_for_loss.predict_proba(X_test_sc)

loss_lr = log_loss(y_test, y_prob_lr)
print(f"Log Loss — Logistic Regression : {loss_lr:.4f}")

# Compare with XGBoost
xgb_model.fit(X_train, y_train)
y_prob_xgb = xgb_model.predict_proba(X_test)
loss_xgb = log_loss(y_test, y_prob_xgb)
print(f"Log Loss — XGBoost             : {loss_xgb:.4f}")

---
## 9. ROC Curve — Multiple Models

### What is ROC-AUC?

The **ROC curve** (Receiver Operating Characteristic) plots **True Positive Rate** (recall for defaulters) against **False Positive Rate** (false alarm rate for non-defaulters) at every possible classification threshold.

**AUC** (Area Under the Curve) summarises the curve into one number:
- AUC = 1.0 → perfect classifier
- AUC = 0.5 → random guess (diagonal)
- AUC 0.70–0.80 → acceptable for credit risk
- AUC > 0.80 → good discriminatory power

### Why use it in ML?

- **Threshold-independent** — evaluates model quality across all operating points, not just at 0.5.
- Unaffected by class imbalance in the way accuracy is.
- A single AUC number allows direct comparison between fundamentally different model families.

### How does it benefit credit risk prediction?

- **Regulatory requirement**: Basel II Pillar 2 and EBA RTS on the IRB approach require ROC-AUC reporting as part of model validation documentation.
- Plotting multiple models on one ROC chart enables visual comparison for thesis figures — each curve's distance from the diagonal shows discriminatory power.
- Risk managers use the ROC curve to select the operating threshold based on business context: a conservative lender accepts lower TPR (approves fewer loans) in exchange for lower FPR (fewer bad loans approved); the curve shows this trade-off explicitly.

In [ ]:
models_for_roc = {
    "Logistic Regression": (lr_for_loss,  X_test_sc),
    "Random Forest"      : (RandomForestClassifier(n_estimators=100, class_weight="balanced",
                             random_state=RANDOM_STATE).fit(X_train, y_train), X_test),
    "XGBoost"            : (xgb_model, X_test),
    "Gradient Boosting"  : (gb_model,  X_test),
}

plt.figure(figsize=(8, 6))
for name, (mdl, Xte) in models_for_roc.items():
    y_prob = mdl.predict_proba(Xte)[:, 1]
    auc    = roc_auc_score(y_test, y_prob)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name}  (AUC={auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random (AUC=0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Credit Risk Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

---
## 10. Hyperparameter Tuning

**Why hyperparameter tuning matters:**

Default model parameters are general-purpose starting points, not optimised for any specific dataset. Tuning finds the configuration that maximises performance on this data. Three strategies are compared here — each with different trade-offs between search efficiency and solution quality.

**All three use `scoring="roc_auc"`** — accuracy would be misleading here due to class imbalance. ROC-AUC is the correct objective for credit risk model selection.

---
### 10a. Random Search — Random Forest

### What is Random Search?

**RandomizedSearchCV** samples `n_iter` random combinations from the parameter distributions and evaluates each with k-fold cross-validation. Unlike Grid Search (which exhaustively tests every combination), Random Search covers the space more efficiently — especially when only a few parameters significantly affect performance.

Bergstra & Bengio (2012) showed that Random Search finds equally good or better solutions than Grid Search in the same compute budget, because most hyperparameter spaces have low effective dimensionality.

### Why use it in ML?

- Much faster than Grid Search for large parameter spaces — a 4-parameter grid with 10 values each = 10,000 combinations; Random Search tests 20.
- `scipy.stats.randint` generates continuous integer ranges so the search is not constrained to a fixed grid.
- `n_jobs=-1` parallelises all CV folds across CPU cores.

### How does it benefit credit risk prediction?

- Random Forest has 4+ interacting parameters (`n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`). Grid Search over these would require hundreds of fits; Random Search finds a good region in 20.
- The best estimator from `random_search.best_estimator_` can be passed directly to `evaluate_model` — no manual reconstruction needed.
- `cv=5` with stratification (default in sklearn for classifiers) preserves the 22%/78% split in each fold — essential for reliable AUC estimates.

### Requirements

- `param_distributions` must use scipy distribution objects or lists — `randint(a, b)` samples integers from `[a, b)`.
- `n_iter`: 20 is a practical starting point; 50–100 gives more thorough coverage at higher cost.
- Feature scaling **not required** for Random Forest.

In [ ]:
rf = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)

param_dist = {
    "n_estimators"     : randint(50, 301),
    "max_depth"        : randint(2, 21),
    "min_samples_split": randint(2, 11),
    "min_samples_leaf" : randint(1, 6),
}

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="roc_auc",      # AUC is a better scoring metric than accuracy here
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

random_search.fit(X_train, y_train)

print("Best params (Random Search):")
print(random_search.best_params_)
print("Best CV ROC-AUC:", round(random_search.best_score_, 4))

evaluate_model(random_search.best_estimator_, X_train, y_train, X_test, y_test,
               "Random Search — Best RF")

---
### 10b. Genetic Algorithm

### What is a Genetic Algorithm?

A **Genetic Algorithm** (GA) is an evolutionary search method inspired by natural selection. It maintains a **population** of candidate hyperparameter configurations (individuals), evaluates each one's **fitness** (CV AUC), selects the best performers as **parents**, combines them via **crossover**, and introduces random **mutations** — then repeats for multiple **generations**.

Five key operations:
1. `create_population` — random initialisation of `pop_size` individuals
2. `fitness_function` — 3-fold CV AUC of the RF with those hyperparameters
3. `selection` — keep top `num_parents` performers
4. `crossover` — randomly inherit each parameter from one of two parents
5. `mutation` — randomly change a parameter with probability `mutation_rate`

### Why use it in ML?

- Explores the parameter space more intelligently than Random Search — later generations concentrate sampling near already-good configurations.
- Works well with **discrete and integer** parameter spaces where gradient-based methods cannot be applied.
- Does not require a differentiable objective — fitness can be any scoring function.
- Introduces diversity through mutation, avoiding premature convergence to local optima.

### How does it benefit credit risk prediction?

- Credit risk model selection involves integer parameters (`n_estimators`, `max_depth`) with non-convex AUC surfaces — GA is well-suited to this topology.
- For the thesis, GA represents the **"advanced search"** tier between Random Search (simple) and Bayesian Optimisation (principled probabilistic) — illustrating the trade-off between exploration and exploitation.
- Mutation rate of 0.2 (20%) prevents the population from converging too quickly to a single parameter region, which is important when the AUC landscape has multiple near-optimal plateaus.

### Requirements

- `pop_size` × `generations` × `cv_folds` model fits total (e.g. 10 × 5 × 3 = 150 fits).
- `mutation_rate`: 0.1–0.3 typical. Too low → premature convergence; too high → random search.
- `random.seed` should be set for reproducibility — the `RANDOM_STATE` constant controls the RF but not the GA's own `random.randint` calls.

In [ ]:
param_space = {
    "n_estimators"     : (50, 200),
    "max_depth"        : (2, 20),
    "min_samples_split": (2, 10),
    "min_samples_leaf" : (1, 5),
}

def create_individual():
    return {k: random.randint(v[0], v[1]) for k, v in param_space.items()}

def create_population(pop_size):
    return [create_individual() for _ in range(pop_size)]

def fitness_function(individual):
    model = RandomForestClassifier(
        **individual,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring="roc_auc", n_jobs=-1)
    return scores.mean()

def evaluate_population(population):
    return [(ind, fitness_function(ind)) for ind in population]

def selection(scored_pop, num_parents):
    scored_pop = sorted(scored_pop, key=lambda x: x[1], reverse=True)
    return [ind for ind, _ in scored_pop[:num_parents]]

def crossover(p1, p2):
    return {k: random.choice([p1[k], p2[k]]) for k in p1}

def mutation(individual, rate=0.1):
    mutated = individual.copy()
    for k, (lo, hi) in param_space.items():
        if random.random() < rate:
            mutated[k] = random.randint(lo, hi)
    return mutated

In [ ]:
def genetic_algorithm(pop_size=10, generations=5, num_parents=4, mutation_rate=0.2):
    population  = create_population(pop_size)
    best_ind, best_score = None, -1

    for gen in range(generations):
        scored = evaluate_population(population)
        scored = sorted(scored, key=lambda x: x[1], reverse=True)

        cur_ind, cur_score = scored[0]
        print(f"Generation {gen+1}  |  best AUC: {cur_score:.4f}  |  params: {cur_ind}")

        if cur_score > best_score:
            best_score, best_ind = cur_score, cur_ind

        parents = selection(scored, num_parents)
        next_pop = parents.copy()
        while len(next_pop) < pop_size:
            child = crossover(random.choice(parents), random.choice(parents))
            next_pop.append(mutation(child, mutation_rate))
        population = next_pop

    return best_ind, best_score

best_params_ga, best_cv_auc_ga = genetic_algorithm(pop_size=10, generations=5)
print("\nBest params (GA):", best_params_ga)
print("Best CV AUC     :", round(best_cv_auc_ga, 4))

In [ ]:
ga_model = RandomForestClassifier(
    **best_params_ga,
    class_weight="balanced",
    random_state=RANDOM_STATE
)
evaluate_model(ga_model, X_train, y_train, X_test, y_test, "Genetic Algorithm — Best RF")

---
### 10c. Bayesian Optimisation

### What is Bayesian Optimisation?

**Bayesian Optimisation** uses a **probabilistic surrogate model** (Gaussian Process in `skopt`) to approximate the objective function (CV AUC as a function of hyperparameters). After each evaluation, it updates its belief about which regions of the parameter space are likely to yield higher scores, and uses an **acquisition function** to select the next point to evaluate — balancing exploration (trying unknown regions) and exploitation (refining known good regions).

`BayesSearchCV` wraps this into a familiar sklearn CV interface, making it a drop-in replacement for `RandomizedSearchCV`.

### Why use it in ML?

- More sample-efficient than Random Search — each evaluation informs the next, so fewer total fits are needed to find a near-optimal configuration.
- Particularly effective when model training is expensive (large datasets, deep models) and each evaluation costs significant time.
- `Integer(a, b)` from `skopt.space` defines continuous integer ranges with proper Gaussian Process handling — unlike `randint` which only samples discretely.

### How does it benefit credit risk prediction?

- With 30,000 rows and 5-fold CV, each RF fit takes several seconds. Bayesian Optimisation finds the best configuration in 30 iterations vs the hundreds Grid Search would need — directly reducing experiment time in the thesis pipeline.
- The Gaussian Process surrogate model provides **uncertainty estimates** per parameter region — regions with high uncertainty are explored, high-confidence regions are exploited. This is the most principled of the three search strategies.
- In the thesis comparison, Bayesian Optimisation should produce the highest or equal CV AUC of the three methods, with fewer total model fits than Random Search.

### Requirements

- `pip install scikit-optimize` required.
- `n_iter=30` — sufficient for a 4-parameter integer space; increase to 50+ for larger search spaces.
- `skopt.space.Integer` (not `scipy.stats.randint`) must be used for parameter definitions.
- Same `scoring="roc_auc"` and `cv=5` as Random Search for a fair comparison.

In [ ]:
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# ── Bayesian Optimisation with skopt ──────────────────────────────────────
search_spaces = {
    "n_estimators"     : Integer(50, 200),
    "max_depth"        : Integer(2, 20),
    "min_samples_split": Integer(2, 10),
    "min_samples_leaf" : Integer(1, 5),
}

bayes_search = BayesSearchCV(
    estimator=RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    search_spaces=search_spaces,
    n_iter=30,
    cv=5,
    scoring="roc_auc",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

bayes_search.fit(X_train, y_train)

print("Best params (Bayesian):")
print(bayes_search.best_params_)
print("Best CV ROC-AUC:", round(bayes_search.best_score_, 4))

In [ ]:
bo_model = bayes_search.best_estimator_
evaluate_model(bo_model, X_train, y_train, X_test, y_test, "Bayesian Optimisation — Best RF")

---
## 11. Summary Comparison

### How to interpret this table

When selecting the best model configuration for the credit risk project, evaluate across three dimensions:

1. **ROC-AUC** — primary performance metric; required for Basel II IRB model validation. Measures how well the model separates defaulters from non-defaulters across all thresholds.
2. **Macro F1** — confirms the model is capturing default cases (minority class), not just performing well on the majority. A model with high AUC but low Macro F1 may still be ignoring defaults.
3. **Accuracy** — included for reference only. Should not be used for model selection on imbalanced data.

**Expected ranking:** Bayesian Optimised RF ≥ Genetic Algorithm RF ≥ Random Search RF > XGBoost ≥ Stacking > ensemble baselines.

**Key thesis insight:** If the AUC gap between Bayesian Optimisation and Random Search is small (< 0.005), the simpler Random Search is preferable in production due to lower computational cost and easier documentation for regulators.

In [ ]:
# Re-run all tuned models and collect ROC-AUC for comparison
summary_models = {
    "Bagging (LR)"         : (bagging_lr,  X_test_sc),
    "XGBoost"              : (xgb_model,   X_test),
    "Gradient Boosting"    : (gb_model,    X_test),
    "Stacking RF+XGB->LR"  : (stack_clf,   X_test),
    "Soft Voting"          : (soft_voting,  X_test_sc),
    "Random Search RF"     : (random_search.best_estimator_, X_test),
    "Genetic Algorithm RF" : (ga_model,    X_test),
    "Bayesian Optimised RF": (bo_model,    X_test),
}

results = []
for name, (mdl, Xte) in summary_models.items():
    y_prob = mdl.predict_proba(Xte)[:, 1]
    y_pred = mdl.predict(Xte)
    results.append({
        "Model"   : name,
        "ROC-AUC" : round(roc_auc_score(y_test, y_prob), 4),
        "Macro-F1": round(f1_score(y_test, y_pred, average="macro"), 4),
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
    })

summary_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).reset_index(drop=True)
print(summary_df.to_string(index=False))